# 5. FastAPI로 감싸기

**목표** — 노트북 01~04에서 한 호출을 **HTTP API**로 만든다. 노트북 안에서만 돌던 코드가 다른 프로그램도 부를 수 있는 서비스가 된다.

**소요 시간** 약 60분

| 다루는 것 | |
| --- | --- |
| 1 | 왜 API로 감싸는가 |
| 2 | 최소 앱 — `/health` |
| 3 | `/ask` 만들기 |
| 4 | 잘못된 값은 서버가 막는다 |
| 5 | 진짜 서버로 띄우기 |
| 6 | 연습문제 |

> **오늘의 마지막 조각이다.** 01의 호출, 02의 토큰, 03의 파라미터가 여기서 엔드포인트 하나로 합쳐진다.

## 0. 준비

In [4]:
import os

from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

load_dotenv(override=True)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.1-flash-lite"

print("준비 완료")

c:\study\llm-api\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


준비 완료


## 1. 왜 API로 감싸는가

지금까지는 노트북에서 직접 `client.models.generate_content(...)`를 불렀다. 이 방식의 한계는 분명하다.

| 한계 | 설명 |
| --- | --- |
| 나만 쓸 수 있다 | 내 PC에서 노트북을 열어야 한다 |
| 키가 노출된다 | 웹 화면이나 앱에 `GEMINI_API_KEY`를 넣을 수 없다 |
| 규칙을 강제할 수 없다 | `temperature=99` 같은 값도 그냥 들어간다 |

**API로 감싸면** 키는 서버에만 두고, 클라이언트는 정해진 주소만 부른다.

```
브라우저 · 모바일 앱 · 다른 서버
        |  HTTP 요청 (질문만)
   FastAPI 서버   <- 키를 여기에만 둔다. 값 검사도 여기서 한다
        |
   Gemini API
```

> 오늘 만드는 `/ask`는 **17일차 `/chat`의 축소판**이다. 17일차에는 여기에 대화 이력과 DB 저장이 붙는다.

## 2. 최소 앱 — `/health`

`FastAPI()` 객체를 만들고 함수 위에 데코레이터를 붙이면 그게 엔드포인트가 된다.

서버를 따로 띄우지 않고 **`TestClient`로 노트북 안에서 바로 호출**해본다. 진짜 서버는 5절에서 띄운다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
app = FastAPI(title="llm-api-basic", version="0.1.0")


@app.get("/health")
def health():
    # 키 값 자체는 절대 돌려주지 않는다. 있는지 여부만 본다.
    return {"status": "ok", "model": MODEL, "key_loaded": bool(os.getenv("GEMINI_API_KEY"))}


test = TestClient(app)
r = test.get("/health")
print(r.status_code, r.json())
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 200 과 JSON 이 돌아오면 성공

In [ ]:
bool()

In [5]:
app = FastAPI(title="llm-api-basic", version="0.1.0")

@app.get("/health")
def health():
    return{"status": "ok", "model": MODEL, "key_loaded": bool(os.getenv("GEMINI_API_KEY"))}

In [6]:
t_client = TestClient(app)
res = t_client.get("/health")
print(res.status_code, res.json())

200 {'status': 'ok', 'model': 'gemini-3.1-flash-lite', 'key_loaded': True}


`200`과 함께 JSON이 돌아왔다면 성공이다.

> **참고:** `TestClient`를 만들 때 `StarletteDeprecationWarning`이 뜰 수 있다. 동작에는 문제가 없으니 무시한다.

## 3. `/ask` 만들기

요청과 응답의 모양을 **Pydantic 모델**로 먼저 정한다.

- **요청** — 노트북 03에서 만져본 파라미터가 여기서 API 스펙이 된다. `Field`로 범위를 박아둔다.
- **응답** — 답만 주지 않고 **토큰 사용량도 같이** 내려준다. 노트북 02의 과금이 여기서 응답 필드가 된다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=500)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_output_tokens: int = Field(default=256, ge=1, le=2048)
    system_instruction: str | None = None


class Usage(BaseModel):
    input_tokens: int
    output_tokens: int
    total_tokens: int


class AskResponse(BaseModel):
    answer: str
    model: str
    usage: Usage


@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    config = types.GenerateContentConfig(
        temperature=req.temperature,
        max_output_tokens=req.max_output_tokens,
        system_instruction=req.system_instruction,
    )
    try:
        r = client.models.generate_content(model=MODEL, contents=req.question, config=config)
    except Exception as e:
        # 429는 우리 버그가 아니라 '지금은 처리 불가'라서 500이 아니라 503이다
        if any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")):
            raise HTTPException(status_code=503, detail="지금은 응답할 수 없습니다.") from e
        raise HTTPException(status_code=502, detail=f"{type(e).__name__}: {str(e)[:200]}") from e

    u = r.usage_metadata
    return AskResponse(
        answer=r.text or "",
        model=r.model_version,
        usage=Usage(
            input_tokens=u.prompt_token_count,
            output_tokens=u.candidates_token_count or 0,
            total_tokens=u.total_token_count,
        ),
    )


print("/ask 등록 완료")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: '/ask 등록 완료' 가 출력되면 성공

In [18]:
class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=500)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_output_tokens: int = Field(default=256, ge=1, le=2048)
    system_instruction: str | None = None


class Usage(BaseModel):
    input_tokens: int
    output_tokens: int
    total_tokens: int


class AskResponse(BaseModel):
    answer: str
    model: str
    usage: Usage

In [19]:
@app.post("/ask", response_model=AskResponse)
def ask(req:AskRequest):
    # config 생성
    config = types.GenerateContentConfig(
        temperature=req.temperature,
        max_output_tokens=req.max_output_tokens,
        system_instruction=req.system_instruction
    )
    # llm model 호출 - client.models.generate_content(config)
    res = client.models.generate_content(model=MODEL, contents=req.question,
                                   config=config)
    #응답 메세지 추출
    u = res.usage_metadata
    return AskResponse(
        answer=res.text,
        model=res.model_version,
        usage=Usage(
            input_tokens= u.prompt_token_count,
            output_tokens=u.candidates_token_count,
            total_tokens= u.total_token_count
        )
    )

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
r = test.post(
    "/ask",
    json={"question": "대한민국의 수도는? 한 단어로.", "temperature": 0.0, "max_output_tokens": 50},
)
print(r.status_code, r.json())

r2 = test.post(
    "/ask",
    json={
        "question": "블랙홀이 뭐야?",
        "system_instruction": "초등학생에게 한 문장으로 설명한다.",
        "max_output_tokens": 120,
    },
)
print()
print(r2.json()["answer"])
print("사용량:", r2.json()["usage"])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: usage 에 토큰 세 값이 실려 오는지 본다

In [15]:
res = t_client.post(
    "/ask",
    json={"question":"오늘점심뭐먹을까?", "temperature":0.0, "max_output_tokens": 20}
)

In [16]:
res.status_code, res.json()

(200,
 {'answer': '점심 메뉴 고르는 게 세상에서 제일 어려운 일이죠! 결정에 도움',
  'model': 'gemini-3.1-flash-lite',
  'usage': {'input_tokens': 9, 'output_tokens': 16, 'total_tokens': 25}})

## 4. 잘못된 값은 서버가 막는다

`Field`에 적어둔 범위를 벗어나면, 우리가 코드를 한 줄도 안 써도 FastAPI가 **`422`** 를 돌려준다.

**Gemini를 부르기 전에 막히므로 토큰도 소모되지 않는다.** 돈이 나가기 전에 거르는 것이라 실무에서 중요하다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
bad_requests = [
    {"question": ""},                                  # 빈 질문
    {"question": "안녕", "temperature": 5.0},           # 범위 초과
    {"question": "안녕", "max_output_tokens": 0},       # 1 미만
]

for body in bad_requests:
    r = test.post("/ask", json=body)
    detail = r.json()["detail"][0]
    print(f"{r.status_code}  {detail['loc']}  {detail['msg']}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 세 요청 모두 422 가 나는지 본다

In [17]:
bad_requests = [
    {"question": ""},                                  # 빈 질문
    {"question": "안녕", "temperature": 5.0},           # 범위 초과
    {"question": "안녕", "max_output_tokens": 0},       # 1 미만
]

for body in bad_requests:
    r = t_client.post("/ask", json=body)
    detail = r.json()["detail"][0]
    print(f"{r.status_code}  {detail['loc']}  {detail['msg']}")

422  ['body', 'question']  String should have at least 1 character
422  ['body', 'temperature']  Input should be less than or equal to 2
422  ['body', 'max_output_tokens']  Input should be greater than or equal to 1


### 오류를 HTTP로 번역한다

노트북 02에서 본 에러들을 이제 상태 코드로 바꿔 내려줘야 한다.

| 원래 오류 | 성격 | 내려줄 HTTP |
| --- | --- | --- |
| 잘못된 입력값 | 부르는 쪽 잘못 | `422` (자동) |
| `429` / `503` | 일시적 | `503` |
| 잘못된 키·모델명 | 재시도해도 소용없음 | `502` |

`429`에 `500`을 쓰면 안 된다. **우리 코드의 버그가 아니라 "지금은 처리할 수 없음"** 이기 때문이다.
위 `/ask` 코드의 `except` 블록이 그 번역을 한다.

## 5. 진짜 서버로 띄우기

셀에 흩어놓은 것을 폴더 루트의 **`app.py`** 로 옮긴다. 골격에 `TODO`가 있으니 채운다.

옮겼으면 **터미널에서** 실행한다.

```powershell
cd 0_llm-api
uv run uvicorn app:app --reload
```

브라우저에서 `http://127.0.0.1:8000/docs`를 연다. **Swagger UI**가 뜬다 — FastAPI가 우리 코드에서 문서를 자동으로 만들어준 것이다.

1. `POST /ask`를 클릭해 펼친다
2. **`Try it out`** 을 누른다
3. `Request body`의 `question`을 원하는 질문으로 고친다
4. **`Execute`** 를 누른다
5. `Code`가 `200`이고 `Response body`에 답과 `usage`가 보이면 성공이다

`temperature`에 `5`를 넣고 `Execute`를 눌러 `422`도 확인해본다.

> **12일차 예고** — 이 `uvicorn` 실행과 Swagger UI가 12일차에서 그대로 이어진다. 거기서는 Gemini 대신 **Supabase**를 붙인다.

## 6. 연습문제

### 연습 5-1. 페르소나를 서버가 관리하기

지금은 클라이언트가 `system_instruction` 전문을 보낸다. 이러면 **누구나 지침을 마음대로 바꿀 수 있다.**

서버가 페르소나를 들고 있고, 클라이언트는 **이름만** 보내도록 `POST /ask/persona`를 만든다.
(17일차에서 이 구조가 사용자별 시스템 프롬프트로 확장된다)

In [ ]:
# TODO: 페르소나 2개 이상을 담은 딕셔너리를 서버 쪽에 만든다
PERSONAS = {}

# TODO: 질문과 페르소나 '이름'만 받는 요청 모델을 만든다

# TODO: POST /ask/persona 를 만든다. 기존 ask() 를 재사용할 것
#       없는 이름이 오면 어떻게 할지 정한다 — 조용히 기본값으로 넘기지 말 것

# TODO: 있는 이름과 없는 이름으로 각각 호출해 결과를 비교 출력한다

## 정리

- [ ] 노트북에서 하던 LLM 호출을 HTTP 엔드포인트로 만들었다
- [ ] `Field`로 파라미터 범위를 강제하면 `422`가 자동으로 나간다는 것을 확인했다
- [ ] 잘못된 입력은 Gemini를 부르기 전에 막히므로 토큰이 소모되지 않는다는 것을 안다
- [ ] 토큰 사용량을 응답에 실어 보내는 이유를 설명할 수 있다
- [ ] `429`에 `500`이 아니라 `503`을 쓰는 이유를 안다
- [ ] `uvicorn`으로 서버를 띄우고 Swagger UI에서 호출해봤다

---

## 오늘 배운 것이 어디로 이어지는가

| 오늘 | 다음 |
| --- | --- |
| `/ask`가 질문 하나를 처리한다 | **17일차** `/chat`이 대화 이력까지 받는다 |
| 응답이 노트북을 닫으면 사라진다 | **11·12일차** Supabase에 저장한다 |
| `uvicorn` + Swagger UI | **12일차** 같은 방식으로 Supabase를 붙인다 |

수고했다.